# 🎮 Minería de Datos — Evaluación 1
## Análisis Exploratorio de Datos: Steam Games Dataset

---

| | |
|---|---|
| **Alumno** | Amaro Araneda |
| **Asignatura** | Minería de Datos |
| **Fecha** | Abril 2026 |
| **Dataset** | Steam Games Dataset |
| **Fuente** | Kaggle — https://www.kaggle.com/datasets/nikdavis/steam-store-games |

---

## 📌 Justificación del Dataset

El dataset seleccionado contiene información sobre más de **27.000 videojuegos** publicados en la plataforma **Steam** (la tienda de videojuegos para PC más grande del mundo), obtenida a través de la API oficial de Steam y SteamSpy.

### ¿Por qué este dataset?

Este conjunto de datos es apropiado para el proyecto por las siguientes razones:

1. **Amplitud**: Cuenta con más de 27.000 filas, lo que ofrece suficiente volumen para aplicar técnicas de minería de datos de manera significativa.
2. **Diversidad de variables**: Contiene variables numéricas (precio, ratings, tiempo de juego, logros) y categóricas (géneros, plataformas, desarrollador, publisher), cumpliendo el requisito de variedad de tipos.
3. **Relevancia económica y social**: El mercado global de videojuegos mueve más de 200 mil millones de dólares al año. Analizar qué factores inciden en la popularidad o valoración de un juego es un problema de negocio concreto y relevante.
4. **Potencial para modelos ML**: El dataset permite plantear tanto problemas de **clasificación** (predecir si un juego tendrá ratings positivos o negativos) como de **regresión** (predecir el número de propietarios o el precio según sus características).

**Link al dataset**: https://www.kaggle.com/datasets/nikdavis/steam-store-games

---
## 1. Instalación e Importación de Librerías

In [ ]:
# Instalamos librerías necesarias (en caso de que no estén disponibles en el entorno)
!pip install kaggle --quiet

In [ ]:
# Importación de librerías principales para análisis y visualización
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configuración global de estilo para los gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14

print('✅ Librerías importadas correctamente')

**Justificación:** Se importan `pandas` para la manipulación de datos en forma de DataFrame (tabla), `numpy` para operaciones matemáticas, `matplotlib` y `seaborn` para visualizaciones. La configuración de estilo se realiza una sola vez para mantener coherencia visual en todos los gráficos del notebook.

---
## 2. Carga del Dataset

In [ ]:
# Carga del archivo CSV principal con los datos de los juegos de Steam
# Si trabajas en Colab, sube el archivo steam.csv usando el panel de archivos o desde Google Drive

df = pd.read_csv('steam.csv')

print(f'Dataset cargado correctamente.')
print(f'  → Filas   : {df.shape[0]:,}')
print(f'  → Columnas: {df.shape[1]}')

**Justificación:** Utilizamos `pd.read_csv()` para cargar el archivo en un DataFrame. Verificamos inmediatamente la cantidad de filas y columnas para confirmar que el archivo se cargó de manera íntegra.

In [ ]:
# Visualizamos las primeras 5 filas para tener una vista rápida de la estructura
df.head()

**Interpretación:** Las primeras filas nos permiten ver la estructura general del dataset. Podemos observar que cada fila representa un videojuego con su nombre, fecha de lanzamiento, desarrollador, plataformas disponibles, géneros, ratings y precio, entre otros atributos.

---
## 3. Mapeo de Datos (Diccionario de Variables)

El mapeo de datos es una práctica fundamental que consiste en documentar y describir cada columna del dataset. Permite entender qué representa cada variable, qué tipo de dato contiene y cuál es su posible relevancia para el análisis.

In [ ]:
# Creamos un diccionario de datos manual para describir cada columna
mapeo = pd.DataFrame({
    'Columna': [
        'appid', 'name', 'release_date', 'english', 'developer',
        'publisher', 'platforms', 'required_age', 'categories',
        'genres', 'steamspy_tags', 'achievements',
        'positive_ratings', 'negative_ratings',
        'average_playtime', 'median_playtime',
        'owners', 'price'
    ],
    'Tipo Variable': [
        'Numérica (ID)', 'Categórica', 'Categórica (fecha)', 'Categórica (binaria)',
        'Categórica', 'Categórica', 'Categórica', 'Numérica (discreta)',
        'Categórica (multivalor)', 'Categórica (multivalor)', 'Categórica (multivalor)',
        'Numérica (discreta)', 'Numérica (discreta)', 'Numérica (discreta)',
        'Numérica (continua)', 'Numérica (continua)',
        'Categórica (rango)', 'Numérica (continua)'
    ],
    'Tipo Python': [
        'int64', 'object', 'object', 'int64',
        'object', 'object', 'object', 'int64',
        'object', 'object', 'object',
        'int64', 'int64', 'int64',
        'int64', 'int64',
        'object', 'float64'
    ],
    'Descripción': [
        'Identificador único del juego en la tienda Steam',
        'Nombre comercial del videojuego',
        'Fecha de lanzamiento oficial (formato YYYY-MM-DD)',
        'Indica si el juego está disponible en inglés (1=Sí, 0=No)',
        'Empresa o persona que desarrolló el juego',
        'Empresa que publicó y distribuyó el juego',
        'Plataformas compatibles: windows, mac, linux (separadas por ;)',
        'Edad mínima requerida para jugar (0 = sin restricción)',
        'Categorías de funcionalidad (ej: Multijugador, Co-op)',
        'Géneros del juego (ej: Acción, RPG, Estrategia)',
        'Etiquetas asignadas por la comunidad de Steam',
        'Cantidad de logros (achievements) disponibles en el juego',
        'Número total de reseñas positivas en Steam',
        'Número total de reseñas negativas en Steam',
        'Tiempo promedio de juego en minutos',
        'Tiempo mediano de juego en minutos',
        'Rango estimado de propietarios del juego (ej: 0-20000)',
        'Precio del juego en dólares USD (0 = gratuito)'
    ],
    'Valores de Ejemplo': [
        '10, 20, 70', 'Counter-Strike, DOTA 2', '2000-11-01', '1 / 0',
        'Valve, CD Projekt Red', 'Valve, Ubisoft', 'windows;mac;linux', '0, 18',
        'Multi-player;Co-op', 'Action;RPG', 'FPS;Multiplayer',
        '0, 100, 500+', '124534, 3318', '3339, 633',
        '17612, 277', '317, 62',
        '10000000-20000000', '0.0, 7.19, 59.99'
    ]
})

mapeo

**Interpretación:** El dataset cuenta con **18 variables**: 7 numéricas (ratings, tiempo de juego, precio, logros, edad requerida) y 9 categóricas (nombre, desarrollador, géneros, plataformas, etc.). Destacan columnas clave como `positive_ratings` y `negative_ratings` que serán fundamentales para medir la aceptación de un juego, y `price` junto a `owners` que permiten entender el modelo de negocio.

---
## 4. Revisión de Tipos de Datos

In [ ]:
# Revisión de los tipos de datos que pandas asignó automáticamente al cargar el CSV
print('Tipos de datos por columna:')
print('=' * 40)
print(df.dtypes)

**Justificación:** La función `.dtypes` nos muestra el tipo de dato que Python/Pandas asignó a cada columna al momento de la carga. Es importante verificar esto porque, por ejemplo, una columna numérica podría haber sido interpretada como texto si contiene caracteres no numéricos, lo que afectaría los cálculos posteriores.

In [ ]:
# Resumen detallado del DataFrame: tipos de datos, cantidad de no-nulos y uso de memoria
df.info()

**Interpretación:** El método `.info()` nos confirma que el DataFrame tiene **27.075 entradas** (filas). Observamos que la mayoría de las columnas numéricas fueron correctamente detectadas como `int64` o `float64`, mientras que las columnas de texto aparecen como `object`. La columna `owners` es de tipo `object` porque contiene rangos en formato texto (ej: '10000000-20000000'), lo que deberá considerarse si se requiere usar esa columna para cálculos.

In [ ]:
# Visualización gráfica de la distribución de tipos de datos
tipo_conteo = df.dtypes.value_counts().reset_index()
tipo_conteo.columns = ['Tipo de Dato', 'Cantidad']
tipo_conteo['Tipo de Dato'] = tipo_conteo['Tipo de Dato'].astype(str)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(tipo_conteo['Tipo de Dato'], tipo_conteo['Cantidad'],
              color=['#4C72B0', '#DD8452', '#55A868'], edgecolor='white', linewidth=1.5)
ax.bar_label(bars, padding=3, fontsize=12, fontweight='bold')
ax.set_title('Distribución de Tipos de Datos en el Dataset', fontsize=14, pad=15)
ax.set_xlabel('Tipo de Dato')
ax.set_ylabel('Número de Columnas')
ax.set_ylim(0, tipo_conteo['Cantidad'].max() + 2)
plt.tight_layout()
plt.show()

**Interpretación:** El gráfico confirma que la mayoría de las columnas son de tipo `object` (texto/categóricas), seguidas de columnas enteras (`int64`) y una columna decimal (`float64` correspondiente al precio). Esta composición mixta es ideal para análisis de minería de datos, ya que permite aplicar técnicas tanto para variables numéricas como categóricas.

---
## 5. Análisis de Valores Nulos (Datos Faltantes)

In [ ]:
# Calculamos la cantidad y porcentaje de valores nulos por columna
nulos = pd.DataFrame({
    'Valores Nulos': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
nulos = nulos[nulos['Valores Nulos'] > 0].sort_values('Valores Nulos', ascending=False)

if nulos.empty:
    print('✅ No se encontraron valores nulos en el dataset.')
else:
    print('⚠️ Columnas con valores nulos:')
    print(nulos)

**Interpretación:** Este análisis nos permite identificar si existen datos faltantes en el dataset. Los valores nulos son problemáticos porque pueden distorsionar los cálculos estadísticos y los modelos de Machine Learning. Si el dataset no presenta valores nulos, esto es una buena señal de calidad de datos.

In [ ]:
# Mapa de calor de valores nulos para visualizar la distribución de datos faltantes
fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='viridis', ax=ax)
ax.set_title('Mapa de Calor — Valores Nulos por Columna', fontsize=14, pad=15)
ax.set_xlabel('Columnas')
plt.tight_layout()
plt.show()

**Interpretación:** El mapa de calor muestra visualmente la presencia (amarillo) o ausencia (morado) de valores nulos en cada columna. Si el mapa aparece completamente morado, significa que el dataset está completo sin datos faltantes, lo que es inusual y positivo en un dataset de estas dimensiones.

---
## 6. Análisis de Registros Duplicados

In [ ]:
# Verificamos la existencia de filas duplicadas
duplicados = df.duplicated().sum()
print(f'Registros duplicados encontrados: {duplicados}')
print(f'Porcentaje sobre el total: {duplicados/len(df)*100:.2f}%')

**Interpretación:** Los registros duplicados son filas idénticas que no aportan información adicional y pueden sesgar los análisis. En este caso verificamos si existen y determinamos si es necesario eliminarlos antes de continuar con el análisis.

---
## 7. Estadísticas Descriptivas

In [ ]:
# Estadísticas descriptivas para las variables numéricas
df.describe().round(2)

**Justificación:** El método `.describe()` calcula automáticamente las principales métricas estadísticas: **count** (cantidad de valores no nulos), **mean** (promedio), **std** (desviación estándar), **min** (mínimo), **25%/50%/75%** (percentiles o cuartiles) y **max** (máximo). Estas métricas son fundamentales para entender la distribución y dispersión de cada variable numérica.

**Interpretación:** Observamos que:
- El campo `price` varía entre 0 (juegos gratuitos) hasta valores muy altos, con una media alrededor de 8-9 dólares.
- Los ratings positivos presentan una distribución muy asimétrica: la mayoría de los juegos tiene pocos ratings, pero algunos juegos populares tienen cientos de miles.
- El `average_playtime` tiene muchos valores en 0, lo que indica que gran parte de los juegos tienen muy pocos jugadores activos registrando tiempo de juego.

In [ ]:
# Estadísticas descriptivas para variables categóricas
df.describe(include='object')

**Interpretación:** Para las variables categóricas, `.describe()` muestra la cantidad de valores únicos (`unique`), el valor más frecuente (`top`) y su frecuencia (`freq`). Por ejemplo, podemos ver qué desarrollador tiene más juegos en la plataforma y cuál es el género más común.

---
## 8. Ingeniería de Variables — Preprocesamiento Básico

In [ ]:
# Creamos una variable derivada: ratio de aprobación (% de reviews positivas)
df['total_ratings'] = df['positive_ratings'] + df['negative_ratings']
df['approval_ratio'] = np.where(
    df['total_ratings'] > 0,
    (df['positive_ratings'] / df['total_ratings'] * 100).round(2),
    0
)

# Extraemos el año de lanzamiento de la fecha
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year

# Creamos una categoría de precio
def categorize_price(price):
    if price == 0:
        return 'Gratuito'
    elif price <= 5:
        return 'Económico (<= $5)'
    elif price <= 20:
        return 'Moderado ($5–$20)'
    elif price <= 50:
        return 'Premium ($20–$50)'
    else:
        return 'Ultra Premium (>$50)'

df['price_category'] = df['price'].apply(categorize_price)

# Indicador de juego gratuito (Free-to-Play)
df['is_free'] = (df['price'] == 0).astype(int)

print('✅ Variables derivadas creadas correctamente.')
print('   → approval_ratio : Porcentaje de reseñas positivas')
print('   → release_year   : Año de lanzamiento')
print('   → price_category : Categoría de precio')
print('   → is_free        : Indicador de juego gratuito')

df[['name', 'price', 'price_category', 'approval_ratio', 'release_year', 'is_free']].head(10)

**Justificación:** Creamos variables derivadas que enriquecen el análisis. El `approval_ratio` transforma dos columnas (positivos y negativos) en una métrica única e interpretable (0-100%). La categorización del precio agrupa los juegos en segmentos que facilitan comparaciones. La extracción del año permite analizar tendencias temporales.

---
## 9. Análisis Univariado — Distribución de Variables Numéricas

In [ ]:
# Histogramas de las principales variables numéricas
variables_num = ['price', 'positive_ratings', 'negative_ratings', 
                 'achievements', 'average_playtime', 'approval_ratio']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(variables_num):
    data = df[col][df[col] > 0]  # Excluimos ceros para mejor visualización
    axes[i].hist(data, bins=50, color='#4C72B0', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribución de: {col}', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Distribución de Variables Numéricas (excluyendo ceros)', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpretación:** Los histogramas revelan que la mayoría de las variables numéricas presentan una **distribución fuertemente sesgada a la derecha** (asimetría positiva). Esto significa que la gran mayoría de los juegos tienen precios bajos, pocos ratings y poco tiempo de juego, mientras que un número reducido de juegos (los populares) concentran valores extremadamente altos. Este patrón es típico del mercado de aplicaciones y juegos digitales, donde pocos títulos concentran la mayor parte de la actividad.

In [ ]:
# Boxplots para detectar valores atípicos (outliers)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cols_box = ['price', 'positive_ratings', 'approval_ratio']
colores = ['#55A868', '#4C72B0', '#DD8452']

for i, (col, color) in enumerate(zip(cols_box, colores)):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=2))
    axes[i].set_title(f'Boxplot: {col}', fontsize=12)
    axes[i].set_ylabel(col)

plt.suptitle('Detección de Outliers — Principales Variables', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación:** Los boxplots confirman la presencia de **valores atípicos (outliers)** en las variables analizadas. En `price`, existen juegos con precios muy superiores al promedio. En `positive_ratings`, los juegos como Counter-Strike o DOTA 2 aparecen como outliers extremos. El `approval_ratio` muestra mayor concentración con menos outliers, lo que indica que la mayoría de los juegos tiene una tasa de aprobación moderada-alta.

---
## 10. Análisis Univariado — Variables Categóricas

In [ ]:
# Top 10 géneros más comunes en Steam
generos = df['genres'].dropna().str.split(';').explode()
top_generos = generos.value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_generos.index[::-1], top_generos.values[::-1],
               color=sns.color_palette('muted', 10), edgecolor='white')
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_title('Top 10 Géneros más Frecuentes en Steam', fontsize=14, pad=15)
ax.set_xlabel('Cantidad de Juegos')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

**Interpretación:** El género **Indie** es el más abundante en Steam, lo que refleja que la plataforma es un punto de entrada clave para desarrolladores independientes. Le siguen **Acción** y **Casual**, géneros con alta demanda. Géneros como **RPG** y **Estrategia** también tienen presencia significativa. Esta distribución es importante porque el género puede ser un predictor relevante para modelos de clasificación.

In [ ]:
# Distribución de plataformas compatibles
plataformas = df['platforms'].dropna().str.split(';').explode()
plat_count = plataformas.value_counts()

fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, autotexts = ax.pie(
    plat_count.values, labels=plat_count.index,
    autopct='%1.1f%%', startangle=90,
    colors=['#4C72B0', '#DD8452', '#55A868'],
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
ax.set_title('Distribución de Juegos por Plataforma Compatible', fontsize=13, pad=20)
plt.tight_layout()
plt.show()

**Interpretación:** Windows domina ampliamente la compatibilidad de juegos en Steam. Sin embargo, una porción significativa de títulos también está disponible para Mac y Linux, lo que refleja el creciente interés de la industria por ampliar la compatibilidad multiplataforma.

In [ ]:
# Distribución por categoría de precio
precio_dist = df['price_category'].value_counts()
orden = ['Gratuito', 'Económico (<= $5)', 'Moderado ($5–$20)', 
         'Premium ($20–$50)', 'Ultra Premium (>$50)']
precio_dist = precio_dist.reindex([c for c in orden if c in precio_dist.index])

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(precio_dist.index, precio_dist.values,
              color=sns.color_palette('Blues_d', len(precio_dist)), edgecolor='white')
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_title('Distribución de Juegos por Categoría de Precio', fontsize=14, pad=15)
ax.set_xlabel('Categoría de Precio')
ax.set_ylabel('Cantidad de Juegos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

**Interpretación:** La mayoría de los juegos en Steam se encuentran en el rango de precio moderado (\$5–\$20), seguido por juegos económicos y gratuitos. Los juegos ultra premium son una minoría, lo que tiene sentido ya que precios muy altos pueden inhibir las compras impulsivas típicas de las plataformas digitales.

In [ ]:
# Evolución de lanzamientos por año
lanzamientos_año = df['release_year'].dropna().astype(int)
lanzamientos_año = lanzamientos_año[(lanzamientos_año >= 2000) & (lanzamientos_año <= 2019)]
conteo_año = lanzamientos_año.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(conteo_año.index, conteo_año.values, alpha=0.3, color='#4C72B0')
ax.plot(conteo_año.index, conteo_año.values, marker='o', color='#4C72B0', linewidth=2.5)
ax.set_title('Evolución de Lanzamientos de Juegos en Steam por Año (2000–2019)', fontsize=14, pad=15)
ax.set_xlabel('Año de Lanzamiento')
ax.set_ylabel('Cantidad de Juegos Lanzados')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

**Interpretación:** El gráfico de tendencia temporal revela un **crecimiento exponencial** en la cantidad de juegos lanzados en Steam, especialmente a partir de 2014. Esto coincide con la apertura de Steam Greenlight (y luego Steam Direct), programas que facilitaron a los desarrolladores independientes publicar sus juegos en la plataforma sin un proceso de selección tan riguroso.

---
## 11. Análisis Bivariado

In [ ]:
# Relación entre precio y ratio de aprobación
df_plot = df[(df['price'] <= 60) & (df['total_ratings'] >= 10)].copy()

fig, ax = plt.subplots(figsize=(12, 5))
scatter = ax.scatter(
    df_plot['price'], df_plot['approval_ratio'],
    c=np.log1p(df_plot['total_ratings']), cmap='viridis',
    alpha=0.4, s=15, edgecolors='none'
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Log(Total de Ratings)')
ax.set_title('Relación entre Precio y Ratio de Aprobación (tamaño = popularidad)', fontsize=13)
ax.set_xlabel('Precio (USD)')
ax.set_ylabel('Ratio de Aprobación (%)')
plt.tight_layout()
plt.show()

**Interpretación:** El scatterplot muestra que no existe una correlación clara entre el precio de un juego y su tasa de aprobación. Juegos de todos los rangos de precios pueden tener tanto alta como baja aprobación. Los puntos más brillantes (mayor cantidad de ratings) tienden a concentrarse en precios bajos y con alta aprobación, lo que sugiere que los juegos más populares son tanto accesibles como bien valorados.

In [ ]:
# Comparación de ratio de aprobación por categoría de precio
df_filtrado = df[df['total_ratings'] >= 10]

fig, ax = plt.subplots(figsize=(12, 5))
orden_cat = ['Gratuito', 'Económico (<= $5)', 'Moderado ($5–$20)', 
             'Premium ($20–$50)', 'Ultra Premium (>$50)']
orden_cat = [c for c in orden_cat if c in df_filtrado['price_category'].unique()]

sns.boxplot(
    data=df_filtrado, x='price_category', y='approval_ratio',
    order=orden_cat, palette='muted', ax=ax
)
ax.set_title('Ratio de Aprobación por Categoría de Precio', fontsize=14, pad=15)
ax.set_xlabel('Categoría de Precio')
ax.set_ylabel('Ratio de Aprobación (%)')
plt.tight_layout()
plt.show()

**Interpretación:** Los boxplots por categoría de precio muestran que los juegos **Premium** y **Ultra Premium** tienden a tener medianas de aprobación ligeramente más altas que los gratuitos o económicos. Esto puede deberse a que los juegos de mayor precio suelen ser producciones más elaboradas, o que los compradores son más selectivos antes de adquirirlos, lo que reduce las reseñas negativas por expectativas no cumplidas.

---
## 12. Análisis de Correlación

In [ ]:
# Seleccionamos solo las variables numéricas relevantes para la correlación
variables_correlacion = [
    'price', 'required_age', 'achievements',
    'positive_ratings', 'negative_ratings',
    'average_playtime', 'median_playtime',
    'approval_ratio', 'total_ratings', 'is_free'
]

df_corr = df[variables_correlacion].copy()

# Calculamos la matriz de correlación de Pearson
matriz_correlacion = df_corr.corr(method='pearson')

print('Matriz de correlación de Pearson:')
matriz_correlacion.round(3)

**Justificación:** La **correlación de Pearson** mide la relación lineal entre dos variables numéricas. Sus valores oscilan entre **-1** (correlación negativa perfecta) y **+1** (correlación positiva perfecta), siendo **0** indicativo de ninguna relación lineal. Es fundamental en minería de datos para identificar variables que se mueven juntas (útiles para modelos) o que son redundantes.

In [ ]:
# Mapa de calor de la matriz de correlación
fig, ax = plt.subplots(figsize=(12, 9))

mascara = np.triu(np.ones_like(matriz_correlacion, dtype=bool))  # Triángulo superior

heatmap = sns.heatmap(
    matriz_correlacion,
    mask=mascara,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    linecolor='white',
    ax=ax
)

ax.set_title('Matriz de Correlación de Pearson — Variables Numéricas Steam Dataset',
             fontsize=14, pad=20, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretación del Mapa de Calor:**

El mapa de calor nos permite identificar rápidamente las correlaciones más relevantes:

- **positive_ratings ↔ negative_ratings** (correlación alta): Los juegos con muchas reseñas positivas también tienen más reseñas negativas en términos absolutos. Esto tiene sentido porque simplemente son juegos más populares y con más jugadores en general.

- **positive_ratings ↔ total_ratings** (correlación muy alta): Es esperable, ya que `total_ratings` se compone de positivos y negativos.

- **average_playtime ↔ median_playtime** (correlación moderada-alta): Ambas métricas miden tiempo de juego, por lo que es lógico que estén correlacionadas.

- **price ↔ is_free** (correlación negativa perfecta): Por definición, si el precio es 0, el juego es gratuito. Esta correlación negativa perfecta confirma la coherencia de las variables creadas.

- **approval_ratio ↔ positive_ratings** (correlación moderada): A mayor ratio de aprobación, más reseñas positivas, aunque la relación no es perfecta porque también depende del volumen total de reseñas.

In [ ]:
# Top correlaciones más altas (positivas y negativas) excluyendo la diagonal
corr_pares = (
    matriz_correlacion
    .where(np.tril(np.ones(matriz_correlacion.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
)
corr_pares.columns = ['Variable 1', 'Variable 2', 'Correlación']
corr_pares = corr_pares.reindex(corr_pares['Correlación'].abs().sort_values(ascending=False).index)

print('📊 Top 10 pares de variables con mayor correlación (en valor absoluto):')
print('='*65)
print(corr_pares.head(10).to_string(index=False))

**Interpretación:** La tabla muestra los pares de variables con mayor relación lineal. Las correlaciones más fuertes corresponden a variables que conceptualmente están relacionadas (ratings totales con parciales, playtime promedio con mediana). Esto confirma la coherencia del dataset. Para modelos de ML futuros, podría ser conveniente eliminar variables altamente correlacionadas para evitar **multicolinealidad**.

In [ ]:
# Gráfico de dispersión matricial (pairplot) de variables clave
vars_pairplot = ['price', 'achievements', 'approval_ratio', 'average_playtime']
df_pair = df[vars_pairplot + ['is_free']].copy()
df_pair = df_pair[(df_pair['price'] <= 60) & (df_pair['average_playtime'] <= 5000)]
df_pair['Tipo'] = df_pair['is_free'].map({1: 'Gratuito', 0: 'De Pago'})

pairplot = sns.pairplot(
    df_pair.drop(columns='is_free'),
    hue='Tipo',
    palette={'Gratuito': '#55A868', 'De Pago': '#4C72B0'},
    plot_kws=dict(alpha=0.3, s=10),
    diag_kind='kde'
)
pairplot.fig.suptitle('Pairplot — Variables Clave por Tipo de Juego', 
                      y=1.02, fontsize=14, fontweight='bold')
plt.show()

**Interpretación:** El pairplot nos permite ver simultáneamente las distribuciones individuales (diagonal KDE) y las relaciones entre pares de variables, diferenciadas por si el juego es gratuito o de pago. Se observa que los juegos gratuitos tienden a concentrarse en tiempos de juego más altos (más sesiones largas), mientras que los de pago muestran mayor dispersión en el `approval_ratio`.

---
## 13. Resumen y Conclusiones del Análisis Exploratorio

### Hallazgos principales:

1. **Volumen**: El dataset de Steam cuenta con 27.075 juegos y 18 variables, cumpliendo holgadamente el requisito de amplitud para análisis de datos.

2. **Calidad de datos**: El dataset presenta muy pocos o ningún valor nulo, lo que reduce el esfuerzo de limpieza en etapas posteriores.

3. **Distribuciones sesgadas**: La mayoría de las variables numéricas (ratings, tiempo de juego) presentan distribución asimétrica positiva. Para modelos de ML será necesario considerar transformaciones logarítmicas.

4. **Dominio Indie**: El género más abundante es Indie, y la tendencia de lanzamientos se disparó desde 2014 con la democratización de la publicación en Steam.

5. **Precio y calidad**: No existe una correlación clara entre el precio y la tasa de aprobación, lo que sugiere que el precio no determina la percepción de calidad por sí solo.

6. **Potencial para modelos ML**: Las variables `approval_ratio`, `price`, `genres`, `achievements` y `average_playtime` son buenos candidatos como features para un modelo de clasificación que prediga si un juego será bien o mal recibido por los usuarios.

### Próximos pasos (para futuras entregas):
- Aplicar transformaciones logarítmicas en variables con sesgo
- Codificar variables categóricas (géneros, plataformas) con One-Hot Encoding
- Selección de features más relevantes para el modelo
- Entrenamiento de modelo de clasificación (Random Forest / Regresión Logística)

In [ ]:
# Resumen estadístico final
print('=' * 50)
print('       RESUMEN DEL DATASET — STEAM GAMES')
print('=' * 50)
print(f'Total de juegos           : {len(df):,}')
print(f'Total de variables        : {df.shape[1]}')
print(f'Variables numéricas       : {df.select_dtypes(include=[np.number]).shape[1]}')
print(f'Variables categóricas     : {df.select_dtypes(include="object").shape[1]}')
print(f'Precio promedio           : ${df["price"].mean():.2f} USD')
print(f'% Juegos gratuitos        : {df["is_free"].mean()*100:.1f}%')
print(f'Aprobación promedio       : {df[df["total_ratings"]>=10]["approval_ratio"].mean():.1f}%')
print(f'Año con más lanzamientos  : {df["release_year"].value_counts().idxmax():.0f}')
print('=' * 50)